In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

f:\my_softers\project_IDE\Anconda\envs\jp_layout_pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:',device)
sequence_len = 28
input_size = 28
hidden_size = 128
num_layers = 2
num_classes = 10
batch_size = 100
num_epochs = 2
learning_rate = 0.001
train_data = torchvision.datasets.MNIST(
    root='data',
    train=True,
    transform=transforms.ToTensor(),
    download=False
)
test_data = torchvision.datasets.MNIST(
    root='data',
    train=False,
    transform=transforms.ToTensor()
)


train_loader = torch.utils.data.DataLoader(
                dataset=train_data,
                batch_size=batch_size,
                shuffle=True
            )
val_loader = torch.utils.data.DataLoader(
                dataset=test_data,
                batch_size=batch_size,
                shuffle=False
            )

# Bidirectional recurrent neural network(many-to-one)
class BiRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(BiRNN,self).__init__()
        self.hidden_size=hidden_size
        self.num_layers=num_layers
        self.lstm=nn.LSTM(
            input_size=input_size,   # 输入数据特征数
            hidden_size=hidden_size, # 隐藏层单元数
            num_layers=num_layers,   # lstm的层数
            bias=False,
            batch_first=True,
            bidirectional=True)
        self.fc=nn.Linear(hidden_size*2,num_classes)
        
    def forward(self, x):
        # Set initial states
        #             （ 层数*2 if 双向 层数 ,  batch_size,      隐层大小 ）
        h0 = torch.zeros(self.num_layers*2,    x.size(0),   self.hidden_size)
        c0 = torch.zeros(self.num_layers*2,    x.size(0),   self.hidden_size)
        # out, _ = self.lstm(x,(h0,c0)) # out.shape = x.size(0),x.size(1), hidden_size*2
        out, _ = self.lstm(x,h0)
        
        
        out = self.fc(out[:,-1,:])
        return out
model = BiRNN(input_size,hidden_size,num_layers,num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
# total_step = len(train_loader):
for epoch in range(num_epochs):
    for i, (images,labels) in enumerate(train_loader):
        images = images.reshape(-1,sequence_len,input_size)
        pred = model(images)
        loss = criterion(pred, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if i % 100 == 0:
            print(f'Epoch:{epoch},step:{i},loss:{loss.item()}')
        
    with torch.no_grad():
        correct = 0
        total = 0
        for i, (images,labels) in enumerate(val_loader):  
            images = images.reshape(-1,sequence_len,input_size)
            pred = model(images)     
            _, predicted = torch.max(pred,1)
            total += labels.size(0)
            correct += (predicted==labels).sum().item()
        print(f'Test acc:{correct/total}')
    torch.save(model.state_dict(),'model.ckpt')
    
    

device: cpu


RuntimeError: For batched 3-D input, hx and cx should also be 3-D but got (2-D, 2-D) tensors

In [ ]:
num = 12.3456789
print(f'{num:.2f}')

12.35


In [ ]:
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
lr = optimizer.param_groups[0]['lr']
params = optimizer.param_groups[0]['params']
print(lr)

0.001


In [ ]:
len(optimizer.param_groups)

1

In [ ]:
import torch
from torch import nn

input_size=12
hidden_size=24
num_layers=1
lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
# 调用
batch_size = 1
seq_len = 144
num_layers = 1

x = torch.randn(batch_size,seq_len,input_size) 
h0 = torch.zeros(num_layers, batch_size, hidden_size)
c0 = torch.zeros(num_layers, batch_size, hidden_size)
res1, res2 = lstm(x,(h0,c0))
print(res1.shape)
print(len(res2))
print(res2[0].shape)
print(res2[1].shape)

torch.Size([1, 144, 24])
2
torch.Size([1, 1, 24])
torch.Size([1, 1, 24])


In [ ]:
res1[:,-1,:].shape

torch.Size([1, 24])

In [ ]:
# 与上述例子相同，只是对x的形状有要求，batch_size 由第1维度--调换-->第2维度
import torch
from torch import nn

input_size=12
hidden_size=24
num_layers=1
lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=False)
# 调用
batch_size = 1
seq_len = 144
num_layers = 1

x = torch.randn(seq_len,batch_size,input_size) 
h0 = torch.zeros(num_layers, batch_size, hidden_size)
c0 = torch.zeros(num_layers, batch_size, hidden_size)
res1, res2 = lstm(x,(h0,c0))
print(res1.shape)
print(len(res2))
print(res2[0].shape)
print(res2[1].shape)

torch.Size([144, 1, 24])
2
torch.Size([1, 1, 24])
torch.Size([1, 1, 24])
